In [ ]:
import time
import threading
import sys

# =========================
# GAME SETUP
# =========================

game_matrix = [
    ['⬛','👻','⬛','⬛','⬛','⬛','⬛','⬛','⬛','⬛','⬛','⬛','⬛','⬛','⬛'],
    ['⬛','👨🏻‍🦰','🟨','🟨','🟨','🟨','🟨','⬛','🟨','🟨','🟨','🟨','🟨','⬛','⬛'],
    ['⬛','⬛','⬛','⬛','⬛','⬛','🟨','⬛','🟨','⬛','⬛','⬛','🟨','⬛','⬛'],
    ['🟨','🟨','🟨','⬛','🟨','🟨','🟨','⬛','🟨','⬛','⬛','⬛','🟨','⬛','⬛'],
    ['🟨','⬛','🟨','⬛','🟨','⬛','⬛','⬛','🟨','🟨','🟨','⬛','🟨','⬛','⬛'],
    ['🟨','⬛','🟨','⬛','🟨','⬛','⬛','⬛','⬛','⬛','🟨','⬛','🟨','⬛','⬛'],
    ['🟨','⬛','🟨','⬛','🟨','🟨','🟨','🟨','🟨','⬛','🟨','⬛','🟨','⬛','⬛'],
    ['🟨','⬛','🟨','⬛','⬛','⬛','⬛','⬛','🟨','⬛','🟨','⬛','🟨','⬛','⬛'],
    ['🟨','⬛','🟨','🟨','🟨','🟨','🟨','⬛','🟨','⬛','🟨','⬛','🟨','⬛','⬛'],
    ['🟨','⬛','⬛','⬛','⬛','⬛','🟨','⬛','🟨','⬛','🟨','⬛','🟨','⬛','⬛'],
    ['🟨','⬛','⬛','⬛','🟨','🟨','🟨','⬛','🟨','⬛','🟨','⬛','🟨','⬛','⬛'],
    ['🟨','🟨','🟨','⬛','🟨','⬛','⬛','⬛','🟨','⬛','🟨','⬛','🟨','⬛','⬛'],
    ['⬛','⬛','🟨','⬛','🟨','🟨','🟨','🟨','🟨','⬛','🟨','⬛','🟨','⬛','⬛'],
    ['🟨','🟨','🟨','⬛','⬛','⬛','⬛','⬛','⬛','⬛','🟨','⬛','🟨','⬛','⬛'],
    ['🟨','⬛','⬛','⬛','🟨','🟨','🟨','⬛','🟨','🟨','🟨','⬛','🟨','🟨','🟨'],
    ['🟨','🟨','🟨','🟨','🟨','⬛','🟨','🟨','🟨','⬛','⬛','⬛','⬛','⬛','⬛'],
]

PLAYER = "👨🏻‍🦰"
GHOST = "👻"
BRIDGE = "🟨"
VOID = "⬛"
DEAD = "☠️"

ROWS = len(game_matrix)
COLS = len(game_matrix[0])

# =========================
# HELPER FUNCTIONS
# =========================

def print_matrix():
    print("\n" * 2)
    for row in game_matrix:
        print(" ".join(row))
    print()

def find_player():
    for r in range(ROWS):
        for c in range(COLS):
            if game_matrix[r][c] == PLAYER:
                return r, c
    return None

def is_valid(r, c):
    return 0 <= r < ROWS and 0 <= c < COLS

# =========================
# TIMEOUT INPUT
# =========================

user_input = None

def get_input():
    global user_input
    user_input = input("Move (WASD): ").lower()

def timed_input(timeout=180):
    global user_input
    user_input = None

    thread = threading.Thread(target=get_input)
    thread.daemon = True
    thread.start()
    thread.join(timeout)

    if user_input is None:
        return None
    return user_input

# =========================
# GAME LOOP
# =========================

moves = {
    'w': (-1, 0),
    's': (1, 0),
    'a': (0, -1),
    'd': (0, 1)
}

print("🎮 BRIDGE BOLT STARTS!")
print_matrix()

while True:
    pos = find_player()
    if not pos:
        print("Error: Player not found")
        break

    r, c = pos

    move = timed_input(180)

    # TIMEOUT
    if move is None:
        game_matrix[r][c] = DEAD
        print_matrix()
        print("⏰ Too slow! Game Over.")
        break

    if move not in moves:
        print("Invalid key! Use WASD.")
        continue

    dr, dc = moves[move]
    nr, nc = r + dr, c + dc

    # OUT OF BOUNDS
    if not is_valid(nr, nc):
        print("❌ Fell off the map! Game Over.")
        break

    next_tile = game_matrix[nr][nc]

    # VOID
    if next_tile == VOID:
        print("❌ You stepped into the void! Game Over.")
        break

    # BACKTRACK (ghost trail)
    if next_tile == GHOST:
        print("👻 The ghost caught you! Game Over.")
        break

    # MOVE ON BRIDGE
    if next_tile == BRIDGE:
        game_matrix[r][c] = GHOST
        game_matrix[nr][nc] = PLAYER

    print_matrix()

    # WIN CONDITION (reach last column)
    if nc == COLS - 1:
        print("🏆 Congratulations! You've escaped!")
        break